# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to load, explore, and analyze the FAIR² Croissant dataset for protein abundances and modifications from extracellular vesicles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# Identify all available record sets
print("Available record sets:")
if hasattr(metadata, "record_sets"):
    for rs in metadata.record_sets:
        print(f"- {rs['@id']}: {rs['name']}")
else:
    # Fallback if metadata.record_sets not available, load manually from metadata json
    meta_json = ds.metadata.to_json() if hasattr(ds.metadata, "to_json") else json.loads(str(ds.metadata))
    if 'recordSet' in meta_json and isinstance(meta_json['recordSet'], list) and len(meta_json['recordSet']):
        for rs in meta_json['recordSet']:
            print(f"- {rs['@id']}: {rs.get('name', 'No name')}")
    else:
        print("No record sets found in metadata.")

# For this dataset (as of writing), list all record sets via generator (mlcroissant exposes them even if not top-level in metadata)
print("\nRecord set IDs found by scanning records interface:")
record_set_ids = set()
try:
    # Scan a few records to detect record_set IDs
    for entry in ds._data.record_sets:
        record_set_ids.add(entry['@id'])
        print(f"- {entry['@id']}: {entry.get('name', 'No name')}")
except Exception as e:
    print(f"Unable to extract record set IDs: {e}")

# For demonstration, print up to one example record from each available record set
for record_set_id in list(record_set_ids)[:2]:  # adjust as needed
    print(f"\nFirst record in record set '{record_set_id}':")
    try:
        for rec in ds.records(record_set=record_set_id):
            print({k: v for k, v in rec.items() if not k.startswith('_')})
            break
    except Exception as err:
        print(f"Could not load records for {record_set_id}: {err}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Reference record set and field `@id`s as identified above.

In [ ]:
# For this dataset, the main record set typically contains protein quantification data.
# We'll attempt to identify the principal record set for demonstration.

main_record_set = None
if len(record_set_ids):
    main_record_set = list(record_set_ids)[0]  # Use the first one for demonstration
    print(f"Selected record set: {main_record_set}")
else:
    print("No record sets found.")

dataframes = dict()
if main_record_set:
    records = list(ds.records(record_set=main_record_set))
    df = pd.DataFrame(records)
    dataframes[main_record_set] = df
    print(f"Loaded DataFrame columns for record set '{main_record_set}':\n{df.columns.tolist()}")
    display(df.head())
else:
    df = pd.DataFrame()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, or grouping data by attributes for further analysis.

In [ ]:
import numpy as np

if not df.empty:
    # Automatically select a likely numeric column (e.g., 'Abundance', 'MW', etc.)
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if not numeric_candidates:
        # Try to convert columns to numeric (ignore errors)
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field}")
        threshold = np.nanmean(df[numeric_field]) if np.nanmean(df[numeric_field]) > 0 else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping on another field (e.g., 'Gene'/'Protein'/'description' or similar)
        group_candidates = [col for col in df.columns if ((col != numeric_field) and df[col].dtype == object)]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by '{group_field}': mean of numeric field per group:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
    else:
        print("No numeric fields available for demonstration in this dataset.")
else:
    print("DataFrame is empty; cannot perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if not df.empty and 'numeric_field' in locals():
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot against group field if available
    if 'group_field' in locals():
        plt.figure(figsize=(10, 4))
        subset = df[[group_field, numeric_field]].dropna().groupby(group_field).mean().reset_index()
        subset = subset.sort_values(numeric_field)
        plt.bar(subset[group_field].astype(str).head(20), subset[numeric_field].head(20))
        plt.title(f"Mean {numeric_field} by {group_field} (Top 20)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load and explore a FAIR² dataset defined by a Croissant schema, referencing all entities by their `@id`. We loaded protein quantification data, performed basic EDA such as normalization and grouping by key attributes, and visualized the numeric distributions. For deeper analysis, refer to domain-specific documentation and the Croissant dataset's JSON-LD for precise field semantics and relationships.